# Brussels Traffic Data Processing Pipeline

Modular pipeline - **MEMORY OPTIMIZED** version.

**Key Fix:** Call `attach_zones_to_objects()` ONCE per filtering stage, not in loops.

In [ ]:
%reset -f

In [2]:
# Standard imports
import sys
sys.path.insert(0, '/home/ubuntu/prem')

import pandas as pd
import numpy as np
import gc
from tqdm import tqdm
import geopandas as gpd
from shapely import wkt

In [3]:
# Modular imports
from utils import log_memory, log_df_memory, load_data, save_detection_results
from filters.preprocessing import (
    filter_by_lifetime,
    attach_zones_to_objects,
    apply_footpath_zone_filter,
    compute_polygon_orientation,
    filter_parallel_vehicles,
    filter_static_objects
)
from regions.brussels.zones import get_lane_zones, get_footpath_zones, get_crosswalk_zones
from ssm.utils import load_config, assign_zones_to_vehicles
from ssm.m_drac import ModifiedDRAC
from ssm.spf import SafetyPotentialField

[SSM] Numba enabled with 7 threads


In [ ]:
# Configuration
START_DATE = "2025-06-04"
END_DATE = "2025-06-04"
DATA_DIR = "/home/ubuntu/data/uploads/objects/clean"
OUTPUT_DIR = "results"

config = load_config("/home/ubuntu/prem/config.yaml")

print("="*70)
print("BRUSSELS TRAFFIC ANALYSIS")
print("="*70)
print(f"Date: {START_DATE} to {END_DATE}")
print("="*70)

BRUSSELS TRAFFIC ANALYSIS
Date: 2025-06-04 to 2025-06-04


## 1. Data Loading

In [5]:
print("\nLoading data...")
log_memory("Before loading")

df = load_data(DATA_DIR, START_DATE, END_DATE, dtypes=config['data']['dtypes'])

log_df_memory(df, "Loaded data")
print(f"Loaded {len(df):,} records")
df.reset_index(drop=True, inplace=True)


Loading data...
[MEMORY] Before loading: 229.4 MB


Loading data: 100%|██████████| 336/336 [00:02<00:00, 157.95it/s]


[DF MEMORY] Loaded data: 2996.8 MB (51,514,602 rows)
Loaded 51,514,602 records


## 2. Lifetime Filtering

In [6]:
print("\n" + "="*70)
print("Lifetime Filtering")
print("="*70)

df = filter_by_lifetime(df, config['preprocessing']['lifetime_filter']['min_lifespan'])
log_df_memory(df, "After lifetime filter")


Lifetime Filtering
[lifespan filter] Removed 6,396 short-lived IDs
  Before: 96,158 IDs (51,514,602 rows)
  After: 89,762 IDs (51,306,851 rows)
[DF MEMORY] After lifetime filter: 3376.2 MB (51,306,851 rows)


np.float64(3376.1717977523804)

## 3. Footpath Zone Filtering

In [7]:
print("\n" + "="*70)
print("Footpath Zone Filtering")
print("="*70)

footpath_zones = get_footpath_zones()
zones_df = pd.DataFrame(footpath_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")

# CRITICAL: Call attach_zones_to_objects ONCE
print(f"Attaching footpath zones to {len(df):,} rows...")
log_memory("Before footpath zones")

df = attach_zones_to_objects(df, gdf_zones, how="left", batch_size=100000)

log_memory("After footpath zones")
print(f"✓ Zones attached! Total rows: {len(df):,}")

df = apply_footpath_zone_filter(df)
df = df.drop(columns=['zone'], errors='ignore')
gc.collect()
log_memory("After footpath filter")


Footpath Zone Filtering
Attaching footpath zones to 51,306,851 rows...
[MEMORY] Before footpath zones: 4613.4 MB


Zone assignment batches: 100%|██████████| 514/514 [00:48<00:00, 10.53it/s]


[MEMORY] After footpath zones: 8080.5 MB
✓ Zones attached! Total rows: 51,306,851
[footpath zone] Removed 1334 objects
[MEMORY] After footpath filter: 7775.3 MB


7775.328125

## 4. Crosswalk Zone Filtering

In [8]:
print("\n" + "="*70)
print("Crosswalk Zone Filtering")
print("="*70)

crosswalk_zones = get_crosswalk_zones()
zones_df = pd.DataFrame(crosswalk_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")
gdf_zones["orientation_deg"] = gdf_zones["geometry"].apply(compute_polygon_orientation)

print(f"Attaching crosswalk zones to {len(df):,} rows...")
log_memory("Before crosswalk zones")

df = attach_zones_to_objects(df, gdf_zones, how="left", batch_size=100000)

log_memory("After crosswalk zones")
print(f"✓ Zones attached! Total rows: {len(df):,}")

# Filter parallel vehicles
removed_ids_global = []
df_in_zones = df[df['zone'].notnull()].copy()

for zone_id in df_in_zones['zone'].unique():
    df_zone = df_in_zones[df_in_zones['zone'] == zone_id]
    orientation = gdf_zones[gdf_zones['id'] == zone_id]['orientation_deg'].iloc[0]
    parallel_ids, _ = filter_parallel_vehicles(df_zone, orientation, threshold=4.0)
    removed_ids_global.extend(parallel_ids)

df = df[~df['id'].isin(removed_ids_global)]
print(f"[crosswalk] Removed {len(removed_ids_global):,} parallel vehicles")

df = df.drop(columns=['zone'], errors='ignore')
gc.collect()
log_memory("After crosswalk filter")


Crosswalk Zone Filtering
Attaching crosswalk zones to 50,311,538 rows...
[MEMORY] Before crosswalk zones: 7775.4 MB


Zone assignment batches: 100%|██████████| 504/504 [00:42<00:00, 11.80it/s]


[MEMORY] After crosswalk zones: 7988.9 MB
✓ Zones attached! Total rows: 50,311,538
[crosswalk] Removed 5 parallel vehicles
[MEMORY] After crosswalk filter: 7749.0 MB


7749.00390625

## 5. Static Object Removal

In [9]:
print("\n" + "="*70)
print("Static Object Removal")
print("="*70)

df = filter_static_objects(df, 
    static_threshold=config['preprocessing']['static_filter']['min_speed'],
    static_ratio_min=0.8)

log_df_memory(df, "After static filter")


Static Object Removal


[static filter] Found 11,558 static objects
[static filter] Removed 11,558 static objects
  Before: 88,423 IDs (50,306,083 rows)
  After: 76,865 IDs (24,277,923 rows)
[DF MEMORY] After static filter: 1597.6 MB (24,277,923 rows)


np.float64(1597.5729818344116)

## 6. Lane Zone Assignment

In [10]:
print("\n" + "="*70)
print("Lane Zone Assignment")
print("="*70)

lane_zones = get_lane_zones()
df = assign_zones_to_vehicles(df, lane_zones)

print(df['zone'].value_counts())
print(f"\nVehicles in lanes: {(df['zone'] != 'unknown').sum():,}")

df_lanes = df[df['zone'] != 'unknown'].copy()
log_df_memory(df_lanes, "Lane vehicles")


Lane Zone Assignment

Assigning zones to 24,277,923 vehicles using spatial join...
Processing in batches of 100,000 rows


Zone assignment: 100%|██████████| 243/243 [00:34<00:00,  7.02it/s]



✓ Zone assignment complete!
  Vehicles in zones: 7,118,225
  Vehicles outside zones: 17,775,974
zone
unknown    17775974
E-L1        2163257
B-L2        1739941
E-L2         882040
C-L1         805308
B-L1         616276
C-L2         419701
D-L1         389443
D-L2          57163
A-L1          45096
Name: count, dtype: int64

Vehicles in lanes: 7,118,225
[DF MEMORY] Lane vehicles: 828.2 MB (7,118,225 rows)


np.float64(828.1931400299072)

## 7. M-DRAC Detection

In [11]:
print("\n" + "="*70)
print("M-DRAC Detection")
print("="*70)

# Generate base pairs first (EXACT OLD CODE WORKFLOW)
from ssm.utils import find_all_nearby_pairs, get_mdrac_pairs

print("\nGenerating nearby pairs...")
log_memory("Before pair generation")

# OLD code signature: find_all_nearby_pairs(df, config)
base_pairs = find_all_nearby_pairs(df_lanes, config)

print(f"✓ Generated {len(base_pairs):,} base pairs")
log_memory("After pair generation")

# Filter pairs for M-DRAC (skip_pair_generation=True means use pre-generated pairs)
print("\nFiltering pairs for M-DRAC...")
mdrac_pairs = get_mdrac_pairs(base_pairs, config, skip_pair_generation=True)
print(f"✓ M-DRAC pairs after filtering: {len(mdrac_pairs):,}\"")

# Detect conflicts from filtered pairs
print("\nDetecting M-DRAC conflicts...")
mdrac_detector = ModifiedDRAC(config)
mdrac_conflicts = mdrac_detector.detect(mdrac_pairs, is_pairs_data=True)

print(f"\n{'='*70}")
print(f"M-DRAC Conflicts: {len(mdrac_conflicts):,}")
print(f"{'='*70}")

if len(mdrac_conflicts) > 0:
    print("\nZone Distribution:")
    print(mdrac_conflicts['zone'].value_counts())
    print("\nTop 5 Critical:")
    print(mdrac_conflicts.nlargest(5, 'MDRAC')[['timestamp', 'zone', 'MDRAC']])


M-DRAC Detection

Generating nearby pairs...
[MEMORY] Before pair generation: 6958.1 MB
  Filtered vehicles: 2,440,465
  Generating pairs (max_distance=8.0m)...
  Processing 664,836 timestamps (batch_size=5,000)


  Pair generation: 100%|██████████| 133/133 [00:03<00:00, 37.05it/s]


  Applying overlap filter (buffer=0.5m)...
  ✓ Generated 180,454 nearby pairs (after overlap filter)
  ✓ Added zone information (zone1/zone2 columns)
✓ Generated 180,454 base pairs
[MEMORY] After pair generation: 6997.5 MB

Filtering pairs for M-DRAC...
 Approaching pairs: 83,087
  Zone filter (same lane): 35,886 pairs (filtered 47,201 different-lane)
  Lateral filter (<= 3.0m): 20,909 pairs (filtered 14,977 not aligned)
  ✓ Total filtered: 62,178 pairs | Remaining: 20,909 pairs
  Speed diff > 0.5: 11,081
  Final MDRAC pairs: 323
✓ M-DRAC pairs after filtering: 323"

Detecting M-DRAC conflicts...
 Approaching pairs: 323
  Zone filter (same lane): 323 pairs (filtered 0 different-lane)
  Lateral filter (<= 3.0m): 323 pairs (filtered 0 not aligned)
  ✓ Total filtered: 0 pairs | Remaining: 323 pairs
  Speed diff > 0.5: 323
  Final MDRAC pairs: 323

M-DRAC Conflicts: 5

Zone Distribution:
zone
D-L1    2
B-L2    2
C-L1    1
Name: count, dtype: int64

Top 5 Critical:
                       ti

In [12]:
# Save M-DRAC results
if len(mdrac_conflicts) > 0:
    mdrac_path = save_detection_results(mdrac_conflicts, OUTPUT_DIR, 'mdrac', 'brussels', START_DATE)
    print(f"Saved to {mdrac_path}")

✓ Saved 5 conflicts to results/brussels/brussels/mdrac/04/mdrac_04.csv
Saved to results/brussels/brussels/mdrac/04/mdrac_04.csv


## Complete

In [13]:
print("\n" + "="*70)
print("BRUSSELS ANALYSIS COMPLETE")
print("="*70)
print(f"M-DRAC: {len(mdrac_conflicts):,}")
print("="*70)


BRUSSELS ANALYSIS COMPLETE
M-DRAC: 5
